03 — Download the dataset from S3 into the workbench
====================================================
Goal: copy train/ and test/ images from MinIO onto the workbench PVC so
PyTorch ImageFolder can read them locally (fast training I/O).

Source (S3):  s3://muffin-chihuahua/train|test/chihuahua|muffin/
Destination:  /opt/app-root/src/data/muffin-chihuahua/...

We skip keys under models/ (ONNX will live there later).

In [ ]:
import os
from pathlib import Path

import boto3

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
endpoint = os.environ.get(
    "AWS_S3_ENDPOINT",
    "http://minio.lab-minio.svc.cluster.local:9000",
)
access_key = os.environ.get("AWS_ACCESS_KEY_ID", "minio")
secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY", "minio123")
bucket = os.environ.get("AWS_S3_BUCKET", "muffin-chihuahua")

# Workbench persistent home (PVC mount)
local_root = Path("/opt/app-root/src/data/muffin-chihuahua")
local_root.mkdir(parents=True, exist_ok=True)

s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

# ---------------------------------------------------------------------------
# Download every object except model artifacts
# ---------------------------------------------------------------------------
print(f"Downloading from s3://{bucket} → {local_root}")
count = 0

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket):
    for obj in page.get("Contents", []):
        key = obj["Key"]

        # Skip "folder" markers and model files
        if key.endswith("/") or key.startswith("models/"):
            continue

        dest = local_root / key
        dest.parent.mkdir(parents=True, exist_ok=True)
        s3.download_file(bucket, key, str(dest))

        count += 1
        if count % 100 == 0:
            print(f"  downloaded {count} files...")

print(f"\nDone. {count} files under {local_root}")

# Quick sanity counts (expect ~500 train / ~100 test per class in this lab)
for split in ("train", "test"):
    for label in ("chihuahua", "muffin"):
        n = len(list((local_root / split / label).glob("*")))
        print(f"  {split}/{label}: {n}")

print("\nOK — continue with 04_train_resnet18.py")